In [ ]:
import json
import random
import torch
import numpy as np
from transformers import pipeline
import requests
from huggingface_hub import login


In [ ]:
import os

def get_secret(label, fallback_path):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        pass

    if os.path.exists(fallback_path):
        with open(fallback_path) as f:
            return json.load(f)[label]

    raise ValueError(f"Secret '{label}' not found")


credential = get_secret("HF_TOKEN", "/kaggle/input/secrets/secrets.json")
URL = get_secret("APP_PRED_URL", "/kaggle/input/secrets/secrets.json")

login(token=credential)
COMPANY    = "MachineInnovators Inc.,a leader in scalable, production-ready machine learning applications"



In [ ]:
generator = pipeline(
    "text-generation",
    model="google/gemma-4-e2b-it",
    dtype=torch.float16,
    device=0,
    max_length=None,
    max_new_tokens=80,
    do_sample=True,
    temperature=1.2,
    top_p= 0.92,
    top_k=50,
    penalty_repetition = 1.3
)


In [ ]:
sentiments = ["positive", "negative", "neutral"]
n_posts = random.randint(10, 50)

for i in range(n_posts):
    sentiment = random.choice(sentiments)
    prompt = (
        f"Write one short {sentiment} social media post about {COMPANY}. Output only the post text."
    )
    response = generator(
                [[{"role": "user", "content": prompt}]],
            )
    text = response[0][0]["generated_text"][-1]["content"].strip()
    requests.post(URL, json={"text": text}, timeout=15)
    print(f"[{i+1}/{n_posts}] {sentiment}: {text[:60]}...")
    